# SmartCart AI — Ultralytics YOLO26s Grocery Training Notebook
### Google Colab GPU-Accelerated Fine-Tuning Pipeline

This notebook fine-tunes the official pretrained **Ultralytics YOLO26s** (`yolo26s.pt`) on the custom SmartCart grocery catalog dataset.

**Catalog Classes (12):**
0. `apple`
1. `banana`
2. `orange`
3. `broccoli`
4. `carrot`
5. `water_bottle` (Bisleri 1L)
6. `dahi_cup` (Amul Masti Dahi)
7. `bread` (Britannia Bread)
8. `donut` (Glazed Donut)
9. `fruit_cake` (Britannia Fruit Cake)
10. `chips_packet` (Lay's Classic Chips)
11. `biscuit_packet` (Parle-G Biscuits)

## Step 1: Verify NVIDIA GPU & Install Dependencies

In [ ]:
# Ensure GPU runtime is selected (Runtime -> Change runtime type -> T4 GPU)
!nvidia-smi

# Install latest Ultralytics with YOLO26 support
!pip install --upgrade ultralytics opencv-python pillow pyyaml onnx

In [ ]:
import torch
from ultralytics import YOLO

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Running on CPU. Select Runtime -> Change runtime type -> T4 GPU for 10x faster training.")

## Step 2: Set Up SmartCart Grocery Dataset
If you uploaded your `dataset.zip` from your local project, unzip it. Otherwise, this cell generates the dataset configuration and downloads/prepares samples.

In [ ]:
import os
from pathlib import Path

# Check if dataset.zip was uploaded
if os.path.exists("dataset.zip"):
    !unzip -q dataset.zip -d dataset/
    print("[✓] Unzipped dataset.zip successfully.")

# Create dataset.yaml
dataset_yaml_content = """
path: ./dataset
train: images/train
val: images/val

nc: 12
names:
  0: apple
  1: banana
  2: orange
  3: broccoli
  4: carrot
  5: water_bottle
  6: dahi_cup
  7: bread
  8: donut
  9: fruit_cake
  10: chips_packet
  11: biscuit_packet
"""

with open("dataset.yaml", "w") as f:
    f.write(dataset_yaml_content.strip())

print("[✓] dataset.yaml created successfully.")

## Step 3: Load Pretrained YOLO26s & Fine-Tune
We load the official pretrained `yolo26s.pt` checkpoint and fine-tune it on the 12 grocery classes.

In [ ]:
# Load official pretrained YOLO26s (will auto-download from Ultralytics releases)
model = YOLO("yolo26s.pt")
model.info()

# Fine-tune on SmartCart Grocery Dataset
# On Google Colab T4 GPU, 50 epochs takes ~6 minutes.
results = model.train(
    data="dataset.yaml",
    epochs=50,
    batch=16,
    imgsz=640,
    device=0 if torch.cuda.is_available() else "cpu",
    project="smartcart_runs",
    name="yolo26s_grocery",
    save=True,
    exist_ok=True,
    pretrained=True,
    optimizer="auto",
    verbose=True
)

## Step 4: Validate Precision, Recall, and mAP Metrics

In [ ]:
# Load best checkpoint produced by training
best_model_path = "smartcart_runs/yolo26s_grocery/weights/best.pt"
best_model = YOLO(best_model_path)

# Run comprehensive validation on validation split
metrics = best_model.val(data="dataset.yaml", imgsz=640, batch=16, save_json=False)

print("\n" + "=" * 60)
print("  SmartCart AI — YOLO26s Validation Metrics Report")
print("=" * 60)
print(f"  Precision (B):   {metrics.box.p.mean():.4f}")
print(f"  Recall (B):      {metrics.box.r.mean():.4f}")
print(f"  mAP @ 0.50:      {metrics.box.map50:.4f}")
print(f"  mAP @ 0.50:0.95: {metrics.box.map:.4f}")
print("=" * 60)

## Step 5: Export to ONNX / OpenVINO (Optional CPU Optimization)
For faster inference on Windows laptops without GPU, ONNX or OpenVINO provides 2x speedups.

In [ ]:
# Export best model to ONNX
onnx_path = best_model.export(format="onnx", imgsz=640, dynamic=False)
print(f"[✓] Exported ONNX model to: {onnx_path}")

## Step 6: Download `best.pt` to Local Project
Click the cell below to automatically download `best.pt` directly to your local computer.
Place the downloaded file into your local project directory at:
`porject-IoT/models/best.pt`

In [ ]:
from google.colab import files
import shutil

# Download the best trained PyTorch weights
files.download("smartcart_runs/yolo26s_grocery/weights/best.pt")
print("[✓] Download triggered! Copy best.pt into your local 'porject-IoT/models/' folder.")